In [1]:
import numpy as np
import pandas as pd
from sklearn import tree, ensemble
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier




In [2]:
data_df = pd.read_csv('../data/colon_data.csv')

In [3]:
data_df['CLASS'].value_counts()

CLASS
short     166
medium     83
long       49
Name: count, dtype: int64

In [4]:
data_df_copy = data_df.copy()

In [5]:
data_df['CLASS'] = data_df['CLASS'].map({'short': 0, 'medium': 1, 'long': 2})

In [6]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()

categorical_cols = data_df.select_dtypes(include=['object']).columns

# Apply label encoding to each
for col in categorical_cols:
    data_df[col] = label_enc.fit_transform(data_df[col])


/var/folders/xd/4rb35j41385021wtcs07_68m0000gn/T/ipykernel_10131/1829060357.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = data_df.select_dtypes(include=['object']).columns


In [7]:
data_df

,AGE,SEX,TOPOGRAPHY,STAGE,T,N,M,SURGERY,CLASS
0,64,1,6,2,3,3,3,0,1
1,70,1,6,2,3,3,3,0,1
2,76,0,0,2,2,3,3,1,1
3,62,1,0,2,1,3,3,1,0
4,62,1,0,3,1,3,3,1,0
...,...,...,...,...,...,...,...,...,...
293,68,0,6,3,1,3,3,2,0
294,41,0,6,3,1,3,3,2,0
295,73,1,6,2,2,3,3,2,1
296,76,1,6,2,3,3,3,2,2


In [8]:
Xy=np.array(data_df)


In [9]:
X = Xy[:, :-1]
y = Xy[:, -1]

In [10]:
trainsize = int(len(data_df)/2)
trainplusvalsize = int(len(data_df)/4)

X_train = X[:trainsize]
y_train = y[:trainsize]
X_val = X[trainsize:trainsize+trainplusvalsize]
y_val = y[trainsize:trainsize+trainplusvalsize]
X_test = X[trainsize+trainplusvalsize:]
y_test = y[trainsize+trainplusvalsize:]


In [11]:
max_depth = 15
bestdepth = -1
bestscore = 0
min_train_score = 0.7


for i in range(max_depth):
    clf = tree.DecisionTreeClassifier(max_depth=i+1)
    clf.fit(X_train, y_train)
    trainscore = clf.score(X_train, y_train)
    valscore = clf.score(X_val, y_val)  # Use factorized y_val here
    print('Depth:', i+1, 'Train Score:', trainscore, 'Validation Score:', valscore)

    if valscore > bestscore and trainscore >= min_train_score:
        bestscore = valscore
        bestdepth = i+1

print(bestdepth, bestscore)


Depth: 1 Train Score: 0.5771812080536913 Validation Score: 0.6891891891891891
Depth: 2 Train Score: 0.697986577181208 Validation Score: 0.2702702702702703
Depth: 3 Train Score: 0.785234899328859 Validation Score: 0.6891891891891891
Depth: 4 Train Score: 0.8187919463087249 Validation Score: 0.7297297297297297
Depth: 5 Train Score: 0.8456375838926175 Validation Score: 0.7162162162162162
Depth: 6 Train Score: 0.8859060402684564 Validation Score: 0.7297297297297297
Depth: 7 Train Score: 0.9261744966442953 Validation Score: 0.6756756756756757
Depth: 8 Train Score: 0.9530201342281879 Validation Score: 0.7162162162162162
Depth: 9 Train Score: 0.9664429530201343 Validation Score: 0.6486486486486487
Depth: 10 Train Score: 0.9798657718120806 Validation Score: 0.7162162162162162
Depth: 11 Train Score: 0.9932885906040269 Validation Score: 0.7162162162162162
Depth: 12 Train Score: 0.9932885906040269 Validation Score: 0.7162162162162162
Depth: 13 Train Score: 0.9932885906040269 Validation Score: 0.6

In [15]:
clf_new = RandomForestClassifier(max_depth=1000, n_estimators=1000)
clf_new.fit(X_train, y_train)
train_score = clf_new.score(X_train, y_train)
val_score = clf_new.score(X_val, y_val)
test_score = clf_new.score(X_test, y_test)
print('Train Score:', train_score, 'Validation Score:', val_score, 'Test Score:', test_score)


Train Score: 0.9932885906040269 Validation Score: 0.6891891891891891 Test Score: 0.8266666666666667
